In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import joblib


torch.manual_seed(42)
np.random.seed(42)

print("PyTorch version:", torch.__version__)

In [ ]:
# 1.1 Load the dataset
df = pd.read_csv('high_diamond_ranked_10min.csv')  
print("Dataset shape:", df.shape)
print(df.head())
print("\nTarget distribution:\n", df['blueWins'].value_counts())


cols_to_drop = ['gameId', 'redFirstBlood', 'redKills', 'redDeaths', 
                'redGoldDiff', 'redExperienceDiff', 'redTotalGold', 
                'redTotalExperience']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

print("Shape after dropping redundant columns:", df.shape)

X = df.drop('blueWins', axis=1)
y = df['blueWins']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

print("Training set shape:", X_train_tensor.shape)

In [ ]:

class LogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        return self.sigmoid(self.linear(x))

input_dim = X_train.shape[1]
model = LogisticRegression(input_dim)
print(model)

In [ ]:

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)


epochs = 100
train_losses = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    train_losses.append(epoch_loss / len(train_loader))
    
    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {train_losses[-1]:.4f}')

In [ ]:

model.eval()
with torch.no_grad():
    y_pred_prob = model(X_test_tensor)
    y_pred = (y_pred_prob > 0.5).float()


accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 5))


cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')


fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:

torch.save(model.state_dict(), 'lol_logistic_model.pth')
print("Model saved!")


loaded_model = LogisticRegression(input_dim)
loaded_model.load_state_dict(torch.load('lol_logistic_model.pth'))
loaded_model.eval()
print("Model loaded successfully!")

In [ ]:

learning_rates = [0.001, 0.01, 0.05, 0.1]
best_lr = 0.01
best_acc = 0

for lr in learning_rates:
    model_tune = LogisticRegression(input_dim)
    optimizer_tune = optim.Adam(model_tune.parameters(), lr=lr)
    
    
    for epoch in range(30):
        model_tune.train()
        for batch_X, batch_y in train_loader:
            optimizer_tune.zero_grad()
            outputs = model_tune(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer_tune.step()
    
    
    model_tune.eval()
    with torch.no_grad():
        y_pred_tune = (model_tune(X_test_tensor) > 0.5).float()
        acc = accuracy_score(y_test, y_pred_tune)
    
    print(f"LR={lr}: Accuracy={acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        best_lr = lr

print(f"Best learning rate: {best_lr} with accuracy {best_acc:.4f}")

In [ ]:

weights = model.linear.weight.data.abs().squeeze().numpy()
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': weights
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance.head(15))
plt.title('Top 15 Most Important Features')
plt.show()

print("Top 10 features:\n", feature_importance.head(10))

In [ ]:
torch.tensor([1.0, 2.0, 3.0])

In [ ]:
nn.Linear(input_dim, 1)
torch.sigmoid(self.linear(x))
nn.BCELoss()

In [ ]:
optimizer.zero_grad()

In [ ]:
weight_decay=0.01

In [ ]:
optimizer.zero_grad()